In [0]:
# ============================================================
# Day 4 - Validation gate (last task of the job): fail the run if any control fails
# Read-only: no table is written.
# ============================================================
from pyspark.sql import functions as F, Window

GOLD = "capstone9_smartmeter.smart_grid_gold"
FRESHNESS_HOURS = 6
results = []   # (check, status, detail)

# 1. Silver -> Gold reconciliation written by 04 (FIN-01 BLOCKED and TC07-08 WARN are not failures)
t = f"{GOLD}.gold_reconciliation_results"
if not spark.catalog.tableExists(t):
    results.append(("reconciliation", "FAIL", f"{t} not found"))
else:
    failed = [r.check_id for r in spark.table(t).where("status = 'FAIL'").select("check_id").collect()]
    results.append(("reconciliation", "FAIL" if failed else "PASS", f"FAIL checks: {failed}"))

# 2. Gold tests: the latest run of EACH test (TC03, LATE) must have passed every metric
t = f"{GOLD}.gold_idempotency_test"
if not spark.catalog.tableExists(t):
    results.append(("gold tests", "FAIL", f"{t} not found"))
else:
    latest = (spark.table(t)
                   .withColumn("_last", F.max("tested_at").over(Window.partitionBy("test_id")))
                   .where("tested_at = _last")
                   .groupBy("test_id").agg(F.count_if(~F.col("passed")).alias("failed"))
                   .collect())
    bad = {r.test_id: r.failed for r in latest if r.failed}
    status = "FAIL" if bad or not latest else "PASS"
    results.append(("gold tests", status, f"tests={[r.test_id for r in latest]}, failures={bad}"))

# 3. Freshness: latest committed version of fact_daily_meter within FRESHNESS_HOURS
t = f"{GOLD}.fact_daily_meter"
if not spark.catalog.tableExists(t):
    results.append(("freshness", "FAIL", f"{t} not found"))
else:
    h = (spark.sql(f"DESCRIBE HISTORY {t}")
              .agg(F.max("timestamp").alias("last_commit"),
                   ((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.max("timestamp"))) / 3600)
                   .alias("age_hours"))
              .first())
    if h.last_commit is None:
        results.append(("freshness", "FAIL", "empty Delta history"))
    else:
        ok = h.age_hours <= FRESHNESS_HOURS
        results.append(("freshness", "PASS" if ok else "FAIL",
                        f"last commit {h.last_commit} ({h.age_hours:.2f} h ago, limit {FRESHNESS_HOURS} h)"))

display(spark.createDataFrame(results, "check string, status string, detail string"))
failed_checks = [c for c, s, _ in results if s != "PASS"]
assert not failed_checks, f"Validation failed: {failed_checks}"
print("VALIDATION PASS")